<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/IIRSI/Project/Step_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 7. RAG: эмбеддинги, BM25, Qdrant, RRF — подробная методичка

Эта методичка продолжает Этапы 1–6. Предполагается, что у вас:
- `db` и `app` в `healthy`, `/health` возвращает 200;
- JWT-аутентификация работает;
- мультиагентная система реализована, тесты зелёные;
- в БД ≥ 1000 пар TM и ≥ 500 терминов глоссария (Этап 6), данные проверены через SQL.

Если чего-то нет — вернитесь к Этапу 6.

**Цель этапа:** реализовать гибридный поиск (BM25 + Qdrant + RRF), который в Этапе 9 подставит контекст в промпт LLM.

**Недель:** 2.

**Что вы получите в конце этапа:**

- `backend/data/embeddings.py` — синглтон `SentenceTransformer` с ленивой загрузкой.
- `backend/retrieval/bm25_index.py` — `BM25Index` с `initialize`/`search` на `rank_bm25.BM25Okapi`.
- `backend/data/translation_memory.py` — **полный** `TMManager` (создан не полностью в Этапе 6 — там были только CSV/JSON-лоадеры в скриптах; сам `TMManager` появляется здесь), с ленивой per-pair BM25 через `_bm25_dirty`, `insert_in_session` с `ON CONFLICT DO NOTHING RETURNING id`, `add_many`, `index_after_insert`, `search`.
- `backend/retrieval/vector_index.py` — `VectorIndex` с Qdrant-клиентом, детерминированным `uuid5`, batch-upsert, `delete_by_filter`, `query_points`.
- `backend/retrieval/hybrid_retriever.py` — `reciprocal_rank_fusion` и `hybrid_search`.
- Обновлённый `infrastructure/docker-compose.yml` — сервис `qdrant` с named volume и TCP-healthcheck.
- Тесты: `test_bm25.py`, `test_tm_search.py`, `test_rrf.py`, `test_hybrid_search.py`.

**Критерии готовности:**

- [ ] `sentence-transformers` модель загружается один раз (singleton).
- [ ] `BM25Index.search` возвращает `TMEntry` с заполненным `score`, только `score > 0`.
- [ ] `TMManager.add_many` помечает пару в `_bm25_dirty`, но **не** пересобирает BM25 сразу; `search` пересобирает по требованию.
- [ ] `TMManager.insert_in_session` использует `INSERT ... ON CONFLICT (id) DO NOTHING RETURNING id`.
- [ ] `VectorIndex._point_id` — `uuid5` от `source_lang|target_lang|en|tgt` (не `uuid4`).
- [ ] Повторный `add_entries` не создаёт дубликатов в Qdrant.
- [ ] Qdrant недоступен → `add_entries` и `search` возвращают `[]`/warning, без исключений.
- [ ] `reciprocal_rank_fusion` суммирует `1/(k+rank+1)`, дедуплицирует по `(en, tgt, source_lang, target_lang)`, `id` берёт из первого источника, где он есть.
- [ ] Ветка `week-7` создана, коммит сделан.

> **Важно.** Этот этап **не вызывает** LLM. `TMManager.search` работает с BM25 (in-memory), `VectorIndex.search` — с Qdrant. Гибридный поиск `hybrid_search` используется в Этапе 9 (`perform_translation` для режимов `tm_hybrid`, `tm_hybrid_critic`, `tm_hybrid_editor`, `full`). Qdrant — обязательный сервис с этого этапа: если он выключен, `perform_translation` в hybrid-режимах деградирует до BM25 (warning в логе).

---

## 7.0. Подготовка: ветка и зависимости

```powershell
cd D:\RUNG
git checkout -b week-7
```

Проверьте, что в `pyproject.toml` есть:

```toml
qdrant-client = "^1.19.0"
sentence-transformers = "^6.0.0"
rank-bm25 = "^0.2.2"
torch = "^2.13.0"
```

`sentence-transformers` тянет `torch` и `transformers` как зависимости. `rank-bm25` — лёгкий пакет (~20 КБ), реализующий `BM25Okapi`. `qdrant-client` — официальный Python-клиент.

> **Ловушка про qdrant-client.** Версия `^1.19.0` использует `client.query_points(...)` для поиска, а не `client.search(...)`. Старый `search` помечен deprecated и будет удалён. В коде — `query_points`. Если используете старую версию клиента (1.6 или ниже) — получите `AttributeError: 'QdrantClient' object has no attribute 'query_points'`. Проверьте `python -m poetry show qdrant-client`.

---

## 7.1. `backend/data/embeddings.py` — синглтон `SentenceTransformer`

**Что это:** обёртка над `SentenceTransformer` с ленивой загрузкой. Один экземпляр модели на процесс — `embedding_model`.

**Зачем это нужно:** модель `paraphrase-multilingual-MiniLM-L12-v2` весит ~470 МБ. Если загружать её в каждом `EmbeddingModel()` — 470 МБ × N экземпляров, `pytest` съест всю RAM. Class-variables `_model`/`_dimension` — один раз на класс, независимо от количества инстансов.

Создайте `backend/data/embeddings.py`:

```python
# backend/data/embeddings.py
import logging
import sys
from typing import List, Union

from sentence_transformers import SentenceTransformer

logger = logging.getLogger(__name__)

_DEFAULT_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


class EmbeddingModel:
    """Lazy singleton wrapper around SentenceTransformer."""

    _model = None
    _dimension: int | None = None

    def __init__(self, model_name: str = _DEFAULT_MODEL):
        self.model_name = model_name

    def _load(self) -> None:
        if EmbeddingModel._model is not None:
            return

        logger.info("Loading embedding model: %s...", self.model_name)
        model = SentenceTransformer(self.model_name)

        get_dim = getattr(model, "get_embedding_dimension", None)
        if get_dim is None:
            get_dim = model.get_sentence_embedding_dimension

        EmbeddingModel._model = model
        EmbeddingModel._dimension = get_dim()
        logger.info(
            "Embedding model loaded (dim=%d)", EmbeddingModel._dimension
        )

    @property
    def dimension(self) -> int:
        self._load()
        return EmbeddingModel._dimension

    def encode(
        self,
        texts: Union[str, List[str]],
        normalize: bool = True,
    ) -> List[List[float]]:
        self._load()
        if isinstance(texts, str):
            texts = [texts]

        embeddings = EmbeddingModel._model.encode(
            texts,
            normalize_embeddings=normalize,
            show_progress_bar=sys.stdout.isatty(),
        )
        return embeddings.tolist()


embedding_model = EmbeddingModel()
```

### Разбор ключевых решений

**Почему `_model`/`_dimension` — class-variables, а не instance.** `EmbeddingModel._model` — атрибут класса. Если создать `EmbeddingModel()` дважды, оба инстанса увидят одну модель. Instance-variables (`self._model`) создали бы отдельную модель на каждый экземпляр — 470 МБ × N. Class-variables — один раз на процесс.

**Почему проверка `if EmbeddingModel._model is not None: return` в начале `_load`, а не `asyncio.Lock`.** `encode` — синхронный метод. Внутри FastAPI он вызывается из `async with` контекста, но между `embedding_model.encode(...)` разными корутинами может произойти race condition: обе корутины видят `_model is None`, обе начинают загрузку. В реальности этого не происходит, потому что (а) `encode` синхронный — GIL защищает от одновременного входа, (б) первая загрузка происходит при старте (или первом запросе), и все последующие ждут. Но `asyncio.Lock` был бы надёжнее. В референсном коде — без лока, простой `if`. **Ограничение зафиксировано**, в Этапе 8 `hf_llm.py` использует `asyncio.Lock` для той же задачи.

**Почему `get_embedding_dimension` vs `get_sentence_embedding_dimension`.** В sentence-transformers 3.x+ добавили `get_embedding_dimension()` как более общий API. В 2.x был только `get_sentence_embedding_dimension()`. Fallback `getattr(model, "get_embedding_dimension", None)` сначала пробует новый API, потом старый — совместимость с обеими версиями. В `pyproject.toml` — `^6.0.0`, новый API есть, fallback не срабатывает.

**Почему `show_progress_bar=sys.stdout.isatty()`.** В интерактивном терминале (TTY) — полоска прогресса видна, удобно. В Docker без TTY (`docker compose up -d`) — `isatty()` → `False`, полоска не рисуется. Без этого условия в контейнере логи забиваются `\r`-последовательностями.

**Почему `normalize=True` по умолчанию.** `paraphrase-multilingual-MiniLM-L12-v2` обучена на normalized-эмбеддингах; косинусное расстояние между ними эквивалентно евклидову. `normalize_embeddings=True` делает `cos(a, b) = dot(a, b)` — быстрее в Qdrant (`Distance.COSINE`), меньше проблем с численной стабильностью. Отключать `normalize` нужно только если модель не обучена на нормализованных векторах.

**Почему `embedding_model = EmbeddingModel()` на module-level.** Синглтон для всех импортов. `from backend.data.embeddings import embedding_model` — один объект. `Validator` (Этап 5) и `VectorIndex` (7.4) используют его же.

**Почему `encode(texts, normalize=True)` принимает `str | List[str]`.** Для одного запроса (`Validator._semantic_match`, `VectorIndex.search`) удобно передать строку. Для батча (`VectorIndex.add_entries`) — список. Внутри — нормализация `str → [str]`, потом `model.encode(...)`.

**Почему `.tolist()`, а не numpy array.** `model.encode(...)` возвращает `numpy.ndarray`. `.tolist()` — конвертация в `List[List[float]]`, JSON-сериализуемо и не привязано к numpy. Qdrant принимает `List[float]`, SQLAlchemy/JSON — тоже. Numpy-массив в коде создал бы зависимость «везде numpy».

### Проверка

```powershell
python -m poetry run python -c "from backend.data.embeddings import embedding_model; print('dim:', embedding_model.dimension); v = embedding_model.encode('привет'); print('len(v[0]):', len(v[0]))"
```

**Что вы должны увидеть** (первый запуск качает модель ~470 МБ, 30–60 сек):

```
dim: 384
len(v[0]): 384
```

**Второй запуск** — модель кэширована в `~/.cache/huggingface/`, загрузка ~3 сек.

### Коммит

```powershell
git add backend/data/embeddings.py
git commit -m "feat(week-7): add SentenceTransformer singleton"
```

---

## 7.2. `backend/retrieval/bm25_index.py` — BM25-индекс

**Что это:** обёртка над `rank_bm25.BM25Okapi` с `initialize(entries)` и `search(query, top_n)`.

**Зачем это нужно:** BM25 — лексический поиск. Он находит документы, где встречаются те же слова, что в запросе, но не находит перефразировки. В hybrid-поиске (7.5) BM25 дополняет векторный поиск: вектор находит семантически близкое, BM25 — точные лексические совпадения (термины, имена собственные, редкие слова). RRF объединяет их.

Создайте `backend/retrieval/bm25_index.py`:

```python
# backend/retrieval/bm25_index.py
import logging
from typing import List

import numpy as np
from rank_bm25 import BM25Okapi

from backend.core.models import TMEntry

logger = logging.getLogger(__name__)


class BM25Index:
    """In-memory BM25 index for translation memory entries."""

    def __init__(self):
        self.bm25 = None
        self._entries: List[TMEntry] = []
        self._initialized = False

    def initialize(self, entries: List[TMEntry]) -> None:
        if not entries:
            self._initialized = False
            self._entries = []
            self.bm25 = None
            logger.debug("BM25 index cleared (no entries)")
            return

        self._entries = entries
        corpus = [entry.en.lower().split() for entry in entries]
        self.bm25 = BM25Okapi(corpus)
        self._initialized = True
        logger.info("BM25 index initialized with %d entries", len(entries))

    def search(self, query: str, top_n: int = 10) -> List[TMEntry]:
        if not self._initialized or not self._entries or self.bm25 is None:
            return []

        query = query.strip()
        if not query:
            return []

        try:
            tokenized_query = query.lower().split()
            scores = self.bm25.get_scores(tokenized_query)
            top_indices = np.argsort(scores)[::-1][:top_n]

            results = []
            for idx in top_indices:
                if scores[idx] > 0:
                    entry = self._entries[idx]
                    results.append(TMEntry(
                        id=entry.id,
                        en=entry.en,
                        tgt=entry.tgt,
                        source_lang=entry.source_lang,
                        target_lang=entry.target_lang,
                        score=float(scores[idx]),
                    ))
            logger.debug(
                "BM25 search returned %d results for query: %s...",
                len(results), query[:50],
            )
            return results
        except Exception as e:
            logger.error("BM25 search error: %s", e)
            return []
```

### Разбор ключевых решений

**Почему `initialize` пересоздаёт `BM25Okapi` целиком, а не добавляет записи.** `rank_bm25` не поддерживает инкрементального добавления. `BM25Okapi(corpus)` строит IDF-статистику по всему корпусу; новый документ меняет IDF всех термов, поэтому документ нельзя «добавить» — нужен полный rebuild. `TMManager._bm25_dirty` (7.3) минимизирует количество rebuild'ов, но каждый rebuild — O(N).

**Почему `initialize([])` очищает индекс, а не падает.** Пустой корпус — валидное состояние (нет данных для этой пары). `BM25Okapi([])` бросил бы `ZeroDivisionError` при подсчёте IDF. Явная ветка `if not entries` — правильно.

**Почему `corpus = [entry.en.lower().split() for entry in entries]`.** `BM25Okapi` ожидает `List[List[str]]` — список документов, каждый документ — список токенов. `lower().split()` — примитивная токенизация по пробелам. Для 18 языков проекта (включая арабский, таджикский, татарский) `split()` работает: слова разделены пробелами, Unicode учтён. Морфологию не разбираем — BM25 по стемам был бы точнее, но требует pymorphy2/nltk для 18 языков — не стоит усложнения.

**Почему `.lower()`.** `BM25Okapi` case-sensitive: `"Book"` и `"book"` — разные термы. Приведение к нижнему регистру для английского работает; для татарского (где заглавные буквы несут смысл в начале предложения) — приемлемо, потому что запрос и документы приводятся одинаково.

**Почему `get_scores(...)` → `np.argsort(scores)[::-1][:top_n]`.** `BM25Okapi.get_scores(query)` возвращает массив размером N (N документов). `argsort` возвращает индексы в порядке возрастания score. `[::-1]` разворачивает → по убыванию. `[:top_n]` — топ-N. Стандартный numpy-паттерн, работает за O(N log N).

**Почему `if scores[idx] > 0`.** BM25 может вернуть 0 или отрицательные значения (если терм отсутствует в документе). Такие документы нерелевантны — их не включаем. Это фильтр «только реальные совпадения».

**Почему `TMEntry(...)` — копия, а не оригинал.** `entry` из `_entries` — неизменяемый Pydantic-объект. Но нам нужно добавить `score`, которого в исходной модели может не быть (или есть `None`). Создание нового `TMEntry` с `score=float(scores[idx])` — единственный способ. Оригинал в `_entries` остаётся с `score=None`.

**Почему `score: float(scores[idx])`, а не `scores[idx]`.** `numpy.float64` — не `float`. Pydantic-модель `TMEntry.score: Optional[float]` принимает numpy-скаляр через coercion, но JSON-сериализация `numpy.float64` иногда падает (не все JSON-энкодеры его понимают). Явный `float(...)` — безопасно.

**Почему `try/except` вокруг всего `search`.** `BM25Okapi.get_scores` может бросить `ValueError` на пустой запрос (хотя мы проверили `query.strip()`); `np.argsort` — на nan в scores (если корпус содержит пустые документы). Возврат `[]` с логом — лучше, чем 500 клиенту.

**Почему `search` фильтрует `score > 0` **после** `argsort`, а не через `np.where(scores > 0)` до.** `argsort` уже даёт порядок; фильтр `> 0` оставляет только непустые результаты. Если бы фильтровали до `argsort`, потеряли бы порядок. Если бы использовали `np.where` — получили бы индексы, но пришлось бы вручную сортировать. Текущий подход — сортировка + обрезка + фильтр — самый простой.

### Коммит

```powershell
git add backend/retrieval/bm25_index.py
git commit -m "feat(week-7): add BM25Index (rank_bm25 wrapper)"
```

---

## 7.3. `backend/data/translation_memory.py` — `TMManager` с ленивой BM25

**Что это:** полноценный менеджер TM. Инкапсулирует:
- in-memory кэш всех TM-пар (`_tm_cache`);
- per-pair BM25-индексы (`_bm25_indexes: Dict[_Pair, BM25Index]`);
- «грязные» пары для ленивой пересборки (`_bm25_dirty: Set[_Pair]`);
- batch-insert с `ON CONFLICT DO NOTHING RETURNING id`;
- синхронизацию с Qdrant (`add_entries`, `delete_by_filter`).

**Зачем это нужно:** это ядро RAG. В Этапе 9 `perform_translation` для hybrid-режимов вызывает `tm_manager.search(...)` и `vector_index.search(...)`. `TMManager.search` отдаёт BM25-результаты, предварительно пересобрав «грязные» индексы.

Создайте `backend/data/translation_memory.py` — полный код из референса. Ключевые фрагменты:

### Константы и `_entry_id`

```python
# backend/data/translation_memory.py
import asyncio
import hashlib
import logging
from typing import Any, Dict, List, Optional, Set, Tuple

from sqlalchemy import delete, func, select
from sqlalchemy.ext.asyncio import AsyncSession

from backend.core.config import get_settings
from backend.core.models import TMEntry
from backend.db.models import TMDB
from backend.db.session import AsyncSessionLocal
from backend.retrieval.bm25_index import BM25Index
from backend.retrieval.vector_index import get_vector_index

logger = logging.getLogger(__name__)

_Pair = Tuple[str, str]


def _entry_id(entry: TMEntry) -> str:
    """
    Deterministic primary key for a translation-memory row.

    The full source and target text are not embedded in the id: Postgres
    limits a B-tree key to 2704 bytes, and long hadith or book chapters
    would break INSERT. The id is SHA-256 over
    source_lang|target_lang|en|tgt, formatted as
    '<src>|<tgt>|<64 hex chars>' so it stays readable in logs.
    """
    raw = f"{entry.source_lang}|{entry.target_lang}|{entry.en}|{entry.tgt}"
    digest = hashlib.sha256(raw.encode("utf-8")).hexdigest()
    return f"{entry.source_lang}|{entry.target_lang}|{digest}"
```

### `TMManager.__init__` и ленивая загрузка

```python
class TMManager:
    def __init__(self) -> None:
        self._tm_cache: List[TMEntry] = []
        self._bm25_indexes: Dict[_Pair, BM25Index] = {}
        self._bm25_dirty: Set[_Pair] = set()
        self._lock = asyncio.Lock()
        self._loaded = False
```

### `_build_bm25_for_pair` и `_rebuild_bm25_all`

```python
    async def _build_bm25_for_pair(self, pair: _Pair) -> None:
        pair_entries = [
            e
            for e in self._tm_cache
            if e.source_lang == pair[0] and e.target_lang == pair[1]
        ]
        bm25 = BM25Index()
        await asyncio.to_thread(bm25.initialize, pair_entries)
        self._bm25_indexes[pair] = bm25
        self._bm25_dirty.discard(pair)

    async def _rebuild_bm25_all(self) -> None:
        pairs: Dict[_Pair, List[TMEntry]] = {}
        for entry in self._tm_cache:
            pairs.setdefault(
                (entry.source_lang, entry.target_lang), []
            ).append(entry)

        self._bm25_indexes = {}
        for pair, pair_entries in pairs.items():
            bm25 = BM25Index()
            await asyncio.to_thread(bm25.initialize, pair_entries)
            self._bm25_indexes[pair] = bm25

        self._bm25_dirty.clear()
```

### `_ensure_indexes_loaded` vs `_rebuild_all_indexes`

```python
    async def _ensure_indexes_loaded(self) -> None:
        """
        Lazy first load in the request path.

        Does NOT recreate the Qdrant collection. Upsert is idempotent
        because point ids are deterministic (uuid5 over the entry key),
        so two processes starting at the same time cannot destroy each
        other's data — they simply write the same points twice.
        """
        await self._rebuild_bm25_all()
        await asyncio.to_thread(
            get_vector_index().add_entries, self._tm_cache
        )
        logger.info("Indexes ensured for %d entries", len(self._tm_cache))

    async def _rebuild_all_indexes(self) -> None:
        """
        Full rebuild, including dropping the Qdrant collection.

        Only safe when called by a single process. This is the Celery
        ``index_translation_memory`` task path; the request path uses
        ``_ensure_indexes_loaded`` instead.
        """
        await self._rebuild_bm25_all()
        await asyncio.to_thread(get_vector_index().recreate_collection)
        await asyncio.to_thread(
            get_vector_index().add_entries, self._tm_cache
        )
        logger.info("Rebuilt all indexes for %d entries", len(self._tm_cache))

    async def ensure_loaded(self) -> None:
        if self._loaded:
            return
        async with self._lock:
            if self._loaded:
                return
            self._tm_cache = await self._fetch_all_entries()
            await self._ensure_indexes_loaded()
            self._loaded = True

    async def load_all_into_cache(self) -> None:
        """Full reload. Celery-task entry point, not the request path."""
        async with self._lock:
            self._tm_cache = await self._fetch_all_entries()
            await self._rebuild_all_indexes()
            self._loaded = True
```

### `add_many`, `insert_in_session`, `_build_insert_do_nothing_stmt`

```python
    async def add_entry(self, entry: TMEntry) -> None:
        await self.add_many([entry])

    async def add_many(self, entries: List[TMEntry]) -> int:
        if not entries:
            return 0

        await self.ensure_loaded()

        async with AsyncSessionLocal() as session:
            inserted = await self.insert_in_session(entries, session)
            if not inserted:
                logger.debug("add_many: nothing to insert (all duplicates)")
                return 0
            try:
                await session.commit()
            except Exception:
                await session.rollback()
                raise

        await self.index_after_insert(inserted)

        logger.info(
            "add_many: inserted %d entries (skipped %d duplicates)",
            len(inserted), len(entries) - len(inserted),
        )
        return len(inserted)

    async def insert_in_session(
        self,
        entries: List[TMEntry],
        session: AsyncSession,
    ) -> List[TMEntry]:
        if not entries:
            return []

        seen: Set[str] = set()
        unique: List[Tuple[TMEntry, str]] = []
        for entry in entries:
            eid = _entry_id(entry)
            if eid in seen:
                continue
            seen.add(eid)
            unique.append((entry, eid))

        if not unique:
            return []

        values = [
            {
                "id": eid,
                "source_lang": entry.source_lang,
                "target_lang": entry.target_lang,
                "source_text": entry.en,
                "target_text": entry.tgt,
            }
            for entry, eid in unique
        ]

        stmt = self._build_insert_do_nothing_stmt(values)
        result = await session.execute(stmt)
        inserted_ids: Set[str] = set(result.scalars().all())

        inserted: List[TMEntry] = []
        for entry, eid in unique:
            if eid in inserted_ids:
                if entry.id is None:
                    entry.id = eid
                inserted.append(entry)

        return inserted

    @staticmethod
    def _build_insert_do_nothing_stmt(values: List[Dict[str, Any]]):
        url = get_settings().ASYNC_DATABASE_URL
        if url.startswith("postgresql"):
            from sqlalchemy.dialects.postgresql import insert as pg_insert
            return (
                pg_insert(TMDB)
                .values(values)
                .on_conflict_do_nothing(index_elements=[TMDB.id])
                .returning(TMDB.id)
            )
        if url.startswith("sqlite"):
            from sqlalchemy.dialects.sqlite import insert as sqlite_insert
            return (
                sqlite_insert(TMDB)
                .values(values)
                .on_conflict_do_nothing(index_elements=[TMDB.id])
                .returning(TMDB.id)
            )
        raise RuntimeError(
            f"Unsupported database URL for TM insert: {url!r}. "
            "Only postgresql and sqlite are supported."
        )
```

### `index_after_insert`, `delete_entry`, `search`

```python
    async def index_after_insert(self, inserted: List[TMEntry]) -> None:
        if not inserted:
            return

        async with self._lock:
            self._tm_cache.extend(inserted)

            for e in inserted:
                self._bm25_dirty.add((e.source_lang, e.target_lang))

            await asyncio.to_thread(
                get_vector_index().add_entries, inserted
            )

    async def delete_entry(self, entry_id: str) -> None:
        async with AsyncSessionLocal() as session:
            stmt = delete(TMDB).where(TMDB.id == entry_id)
            result = await session.execute(stmt)
            try:
                await session.commit()
            except Exception:
                await session.rollback()
                raise

            if (result.rowcount or 0) == 0:
                logger.warning("TM entry with id %s not found", entry_id)
                return

        async with self._lock:
            removed: Optional[TMEntry] = None
            for i, entry in enumerate(self._tm_cache):
                if entry.id == entry_id:
                    removed = entry
                    del self._tm_cache[i]
                    break

            if removed is None:
                logger.warning("TM entry %s not present in cache", entry_id)
                return

            self._bm25_dirty.add((removed.source_lang, removed.target_lang))
            await asyncio.to_thread(
                get_vector_index().delete_by_filter,
                removed.source_lang,
                removed.target_lang,
                removed.en,
                removed.tgt,
            )

    async def search(
        self,
        text: str,
        source_lang: str,
        target_lang: str,
        top_n: int = 5,
    ) -> List[TMEntry]:
        await self.ensure_loaded()

        pair: _Pair = (source_lang, target_lang)

        if pair in self._bm25_dirty:
            async with self._lock:
                if pair in self._bm25_dirty:
                    await self._build_bm25_for_pair(pair)

        bm25 = self._bm25_indexes.get(pair)
        if bm25 is None:
            return []

        loop = asyncio.get_running_loop()
        try:
            return await loop.run_in_executor(
                None, bm25.search, text, top_n
            )
        except Exception as e:
            logger.error("BM25 search error: %s", e)
            return []
```

### `_fetch_all_entries`, `list_all`, `count`

```python
    async def _fetch_all_entries(self) -> List[TMEntry]:
        async with AsyncSessionLocal() as session:
            result = await session.execute(select(TMDB))
            return [
                TMEntry(
                    id=e.id, en=e.source_text, tgt=e.target_text,
                    source_lang=e.source_lang, target_lang=e.target_lang,
                )
                for e in result.scalars().all()
            ]

    async def list_all(
        self, source_lang: Optional[str] = None,
        target_lang: Optional[str] = None,
        limit: int = 100, offset: int = 0,
    ) -> List[TMEntry]:
        async with AsyncSessionLocal() as session:
            stmt = select(TMDB)
            if source_lang:
                stmt = stmt.where(TMDB.source_lang == source_lang)
            if target_lang:
                stmt = stmt.where(TMDB.target_lang == target_lang)
            stmt = stmt.offset(offset).limit(limit)
            result = await session.execute(stmt)
            return [
                TMEntry(
                    id=e.id, en=e.source_text, tgt=e.target_text,
                    source_lang=e.source_lang, target_lang=e.target_lang,
                )
                for e in result.scalars().all()
            ]

    async def count(
        self, source_lang: Optional[str] = None,
        target_lang: Optional[str] = None,
    ) -> int:
        async with AsyncSessionLocal() as session:
            stmt = select(func.count()).select_from(TMDB)
            if source_lang:
                stmt = stmt.where(TMDB.source_lang == source_lang)
            if target_lang:
                stmt = stmt.where(TMDB.target_lang == target_lang)
            result = await session.execute(stmt)
            return result.scalar_one()


tm_manager = TMManager()
```

### Разбор ключевых решений

**Почему `_entry_id` — SHA-256, а не просто конкатенация.** `<src>|<tgt>|<sha256>` — это идемпотентный ключ для SQL. Текст не в ключе, потому что Postgres ограничивает B-tree 2704 байтами. SHA-256 даёт 64 hex-символа — фиксированная длина. Плюс `<src>|<tgt>|` в начале — человекочитаемый префикс для логов (`ar|ru|3a4f...` сразу видно, какая пара).

**Почему `_tm_cache` — list, а не dict.** Два соображения: (1) BM25Okapi требует `List[TMEntry]` на вход, (2) `list_all` фильтрует по языкам — линейный поиск по 5000 записям быстрый. Если станет медленно — переделаем на `Dict[id, TMEntry]`, но 5000 записей не оправдывают сложности.

**Почему `_bm25_indexes` — `Dict[_Pair, BM25Index]`, а не один общий.** Каждая языковая пара — свой корпус. BM25 для `ar_ru` содержит только арабско-русские пары; `ru_tt` — только русско-татарские. Это правильно: запрос `"كتاب"` (арабский) не должен находить русские документы. Один общий индекс смешал бы IDF между языками, что дало бы шум.

**Почему `_bm25_dirty` — `Set[_Pair]`, а не флаг `is_dirty`.** Пары обновляются независимо. `add_entry` для `ar_ru` не должен инвалидировать `ru_tt`. Множество пар — точный трекер «что пересобрать».

**Почему `_build_bm25_for_pair` **не** блокирует весь `_tm_cache` на rebuild.** Создаёт локальный `BM25Index`, инициализирует его в `asyncio.to_thread` (BM25Okapi — CPU-bound), потом атомарно заменяет `self._bm25_indexes[pair]`. Другие пары не затрагиваются.

**Почему `asyncio.to_thread(bm25.initialize, pair_entries)`.** `BM25Okapi(corpus)` — синхронный CPU-bound. В FastAPI event loop'е это заблокировало бы всё приложение на время построения. `to_thread` — выносит в threadpool, event loop свободен.

**Почему `_ensure_indexes_loaded` **не** пересоздаёт Qdrant-коллекцию, а `_rebuild_all_indexes` — пересоздаёт.** Это **самое важное** решение 7.3:

- `_ensure_indexes_loaded` вызывается из `ensure_loaded` на первом запросе в FastAPI. Может быть несколько FastAPI-воркеров; каждый при первом запросе вызывает эту функцию. Если пересоздавать коллекцию — второй воркер удалит данные первого. **Idempotent upsert** (uuid5 point id) позволяет писать одно и то же дважды без конфликта.

- `_rebuild_all_indexes` вызывается из `load_all_into_cache`, что вызывается из Celery-задачи `index_translation_memory`. Это **single-process** контекст: один Celery-воркер, одна задача. Здесь можно drop + recreate — безопасно.

**Почему `ensure_loaded` под `async with self._lock`, а `_rebuild_bm25_all` — нет.** `_rebuild_bm25_all` вызывается **из** `ensure_loaded` (уже под локом) и **из** `load_all_into_cache` (тоже под локом). Если бы `_rebuild_bm25_all` сам брал лок — deadlock (asyncio.Lock не reentrant). Поэтому лок держится в публичных методах, приватные — без лока.

**Почему в `ensure_loaded` двойная проверка `if self._loaded: return`.** First check — fast path без лока (типичный случай). Second — под локом, на случай race: два запроса одновременно видят `_loaded=False`, оба идут под лок, первый загружает и ставит `True`, второй видит `True` и выходит. Классический double-checked locking.

**Почему `insert_in_session` принимает session снаружи.** В Этапе 10 `ActiveLearningManager.confirm_correction` использует `insert_in_session(entry, session)` в **той же транзакции**, что и `example.status = "confirmed"`. Если бы `insert_in_session` открывал свою сессию и коммитил — атомарность ломалась бы. Внешняя сессия — контроль у caller'а.

**Почему `_build_insert_do_nothing_stmt` выбирает диалект по URL-префиксу.** `pg_insert` и `sqlite_insert` — разные классы с разными методами. `pg_insert(...).on_conflict_do_nothing(index_elements=[TMDB.id]).returning(TMDB.id)` работает в Postgres 9.5+; `sqlite_insert(...)` — в SQLite 3.35+. Проверка `url.startswith("postgresql")` — надёжнее, чем `isinstance(engine.dialect, ...)`, потому что движок может быть не инициализирован.

**Почему `.returning(TMDB.id)`, а не `.rowcount`.** `RETURNING id` возвращает **только реально вставленные** строки. При `ON CONFLICT DO NOTHING` конфликтующие строки не попадают в `RETURNING`. `rowcount` вернул бы «сколько строк обработано», включая конфликтные — бесполезно для счётчика «вставлено N».

**Почему `inserted_ids: Set[str] = set(result.scalars().all())`.** `RETURNING id` возвращает строки; `scalars().all()` — список id. `set` для O(1) lookup. При вставке 1000 записей, из которых 500 дубликатов, `inserted_ids` содержит 500 id, и `inserted` содержит 500 `TMEntry`.

**Почему в `insert_in_session` присваивается `entry.id = eid` для успешно вставленных.** `TMEntry.id` изначально `None` (caller может передать без `id`). После вставки мы знаем реальный `id` (это `_entry_id(entry)`). Присваиваем — чтобы caller мог использовать `entry.id` для последующей работы (например, `delete_entry(entry.id)`).

**Почему `index_after_insert` берёт `self._lock`.** Модифицирует `self._tm_cache` (extend) и `self._bm25_dirty` (add). Оба — shared state. `add_entries` в Qdrant — синхронный, через `to_thread`.

**Почему `index_after_insert` **помечает** пару грязной, а не пересобирает BM25.** Это и есть «ленивая пересборка»: `add_many` вставляет 1000 пар за 1 секунду, потом помечает пару. Первый `search` пересоберёт BM25 — O(N) один раз. Если бы `add_many` пересобирал сразу — 1000 записей × O(1000) = 10⁶ операций токенизации, несколько минут. Ленивая пересборка — ключевая оптимизация.

**Почему `delete_entry` сначала `DELETE FROM tm_db`, потом `_tm_cache`.** Сначала персистентное хранилище (БД), потом in-memory. Если `DELETE` упадёт — in-memory не тронут, состояние консистентно (данные в БД и в кэше совпадают). Обратный порядок (сначала кэш) дал бы рассинхрон при падении DELETE.

**Почему `delete_by_filter` с `(source_lang, target_lang, source_text, target_text)`, а не `delete_by_id(point_id)`.** Qdrant-клиент позволяет удалять по point id (uuid5), но `TMManager` хранит `entry.id` (SHA-256), не `point_id` (uuid5). Можно было бы вычислить `_point_id(entry)` и удалить по нему — но `delete_by_filter` с payload-полями надёжнее: если point id в Qdrant не совпадает с нашим расчётом (старая версия, ручной импорт) — filter всё равно найдёт и удалит. `delete_by_id` промахнулся бы.

**Почему `search` проверяет `pair in self._bm25_dirty` дважды.** First check — fast path (обычно не dirty). Second — под локом, на случай race: два `search` одновременно видят `dirty`, оба идут под лок, первый пересобирает, второй видит `not dirty` (первый сделал `discard`) и выходит. Стандартный double-checked locking.

**Почему `search` использует `loop.run_in_executor(None, bm25.search, text, top_n)`, а не `asyncio.to_thread`.** Оба варианта работают одинаково (`to_thread` — это обёртка над `run_in_executor` с дефолтным executor). В коде — `run_in_executor` для совместимости со старым стилем. `to_thread` появился в Python 3.9. С `python = "^3.12"` — оба доступны.

**Почему `search` возвращает `[]` при `bm25 is None`.** Пары нет в `_bm25_indexes` — значит, для этой пары нет данных в `_tm_cache`. `[]` — правильный ответ.

### Проверка

```powershell
python -m poetry run python -c "import asyncio; from backend.data.translation_memory import tm_manager; async def m(): await tm_manager.ensure_loaded(); print('loaded:', len(tm_manager._tm_cache), 'entries'); print('pairs:', list(tm_manager._bm25_indexes.keys())); r = await tm_manager.search('hello', 'en', 'ru', top_n=3); print('found:', len(r)); asyncio.run(m())"
```

**Что вы должны увидеть** (при наличии данных из Этапа 6, при условии что Qdrant off):

```
... WARNING Qdrant connection failed (attempt 1/5): ...
... (× 5)
... ERROR Could not connect to Qdrant after 5 attempts
... WARNING Qdrant unavailable; skipping vector addition
... INFO Indexes ensured for N entries
loaded: N entries
pairs: [('ar', 'ru'), ('ru', 'tt'), ('tg', 'fa'), ('ru', 'ba')]
... INFO BM25 index initialized with X entries
found: 0 или 1-3 (зависит от данных)
```

`ensure_loaded` загрузил все TM-пары, построил per-pair BM25, попытался загрузить Qdrant (упал, потому что Этап 7.6 ещё не сделан).

### Коммит

```powershell
git add backend/data/translation_memory.py
git commit -m "feat(week-7): add TMManager with lazy per-pair BM25 and ON CONFLICT insert"
```

---

## 7.4. `backend/retrieval/vector_index.py` — Qdrant-индекс

**Что это:** обёртка над `qdrant_client.QdrantClient` с детерминированным `uuid5` point id, batch-upsert, filter-based search и delete.

**Зачем это нужно:** векторный поиск находит семантически близкое, чего не находит BM25 (перефразировки, синонимы, разные языки). Qdrant — быстрая векторная БД с фильтрами по payload (язык пары, source/target text).

Создайте `backend/retrieval/vector_index.py` — полный код из референса:

```python
# backend/retrieval/vector_index.py
import logging
import time
import uuid
from typing import List

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    FieldCondition,
    Filter,
    MatchValue,
    PointStruct,
    VectorParams,
)

from backend.core.config import get_settings
from backend.core.models import TMEntry
from backend.data.embeddings import embedding_model

logger = logging.getLogger(__name__)

_DELETE_RETRIES = 3
_DELETE_RETRY_DELAY = 1

_POINT_NAMESPACE = uuid.UUID("6ba7b810-9dad-11d1-80b4-00c04fd430c8")


def _point_id(entry: TMEntry) -> str:
    key = f"{entry.source_lang}|{entry.target_lang}|{entry.en}|{entry.tgt}"
    return str(uuid.uuid5(_POINT_NAMESPACE, key))


class VectorIndex:
    """Vector index using Qdrant for semantic search."""

    def __init__(
        self,
        host: str = None,
        port: int = None,
        collection_name: str = None,
    ):
        settings = get_settings()
        self.host = host or settings.QDRANT_HOST
        self.port = port or settings.QDRANT_PORT
        self.collection_name = collection_name or settings.QDRANT_COLLECTION_NAME
        self.api_key = settings.QDRANT_API_KEY
        self.client = None

    def _ensure_connection_and_collection(self, retries: int = 5, delay: int = 2):
        if self.client is not None:
            return

        url = f"http://{self.host}:{self.port}"
        logger.info("Connecting to Qdrant at %s", url)

        client_kwargs = {
            "url": url,
            "timeout": 10.0,
            "check_compatibility": False,
        }
        if self.api_key:
            client_kwargs["api_key"] = self.api_key

        for attempt in range(1, retries + 1):
            try:
                self.client = QdrantClient(**client_kwargs)
                self.client.get_collections()
                logger.info("Connected to Qdrant at %s", url)
                self._ensure_collection()
                logger.info("Qdrant collection '%s' ready", self.collection_name)
                return
            except Exception as exc:
                logger.warning(
                    "Qdrant connection failed (attempt %d/%d): %s",
                    attempt, retries, exc,
                )
                self.client = None
                if attempt < retries:
                    time.sleep(delay)

        logger.error("Could not connect to Qdrant after %d attempts", retries)

    def _ensure_collection(self):
        try:
            self.client.get_collection(self.collection_name)
            logger.debug("Collection '%s' already exists", self.collection_name)
        except Exception:
            logger.info("Creating Qdrant collection '%s'", self.collection_name)
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=VectorParams(
                    size=embedding_model.dimension,
                    distance=Distance.COSINE,
                ),
            )
            logger.info("Collection '%s' created", self.collection_name)

    def recreate_collection(self) -> None:
        self._ensure_connection_and_collection()
        if self.client is None:
            logger.warning("Qdrant unavailable; skipping collection recreation")
            return

        try:
            self.client.delete_collection(self.collection_name)
            logger.info("Deleted Qdrant collection '%s'", self.collection_name)
        except Exception as e:
            logger.warning("Delete collection failed: %s", e)

        self._ensure_collection()

    def add_entries(self, entries: List[TMEntry]) -> None:
        if not entries:
            return

        self._ensure_connection_and_collection()
        if self.client is None:
            logger.warning("Qdrant unavailable; skipping vector addition")
            return

        try:
            texts = [entry.en for entry in entries]
            vectors = embedding_model.encode(texts, normalize=True)

            points = []
            for entry, vector in zip(entries, vectors):
                payload = {
                    "source_text": entry.en,
                    "target_text": entry.tgt,
                    "source_lang": entry.source_lang,
                    "target_lang": entry.target_lang,
                }
                if entry.id is not None:
                    payload["id"] = entry.id
                points.append(
                    PointStruct(
                        id=_point_id(entry),
                        vector=vector,
                        payload=payload,
                    )
                )
            self.client.upsert(collection_name=self.collection_name, points=points)
            logger.info("Added %d vectors to '%s'", len(points), self.collection_name)
        except Exception as e:
            logger.error("Failed to add vectors to Qdrant: %s", e)

    def delete_by_filter(
        self,
        source_lang: str,
        target_lang: str,
        source_text: str,
        target_text: str,
    ) -> None:
        self._ensure_connection_and_collection()
        if self.client is None:
            logger.warning("Qdrant unavailable; skipping vector deletion")
            return

        points_filter = Filter(
            must=[
                FieldCondition(key="source_lang", match=MatchValue(value=source_lang)),
                FieldCondition(key="target_lang", match=MatchValue(value=target_lang)),
                FieldCondition(key="source_text", match=MatchValue(value=source_text)),
                FieldCondition(key="target_text", match=MatchValue(value=target_text)),
            ]
        )

        for attempt in range(1, _DELETE_RETRIES + 1):
            try:
                self.client.delete(
                    collection_name=self.collection_name,
                    points_selector=points_filter,
                )
                logger.info(
                    "Deleted vectors for %s -> %s (%s->%s)",
                    source_text[:50], target_text[:50],
                    source_lang, target_lang,
                )
                return
            except Exception as e:
                logger.warning(
                    "Qdrant delete failed (attempt %d/%d): %s",
                    attempt, _DELETE_RETRIES, e,
                )
                if attempt < _DELETE_RETRIES:
                    time.sleep(_DELETE_RETRY_DELAY)

        logger.error(
            "Failed to delete vectors from Qdrant after %d attempts",
            _DELETE_RETRIES,
        )

    def search(
        self,
        query: str,
        source_lang: str,
        target_lang: str,
        top_n: int = 5,
    ) -> List[TMEntry]:
        self._ensure_connection_and_collection()
        if self.client is None:
            logger.warning("Qdrant unavailable; returning empty results")
            return []

        try:
            query_vector = embedding_model.encode(query)[0]
            search_filter = Filter(
                must=[
                    FieldCondition(key="source_lang", match=MatchValue(value=source_lang)),
                    FieldCondition(key="target_lang", match=MatchValue(value=target_lang)),
                ]
            )
            search_result = self.client.query_points(
                collection_name=self.collection_name,
                query=query_vector,
                limit=top_n,
                query_filter=search_filter,
            )
            results = []
            for hit in search_result.points:
                payload = hit.payload or {}
                results.append(
                    TMEntry(
                        id=payload.get("id"),
                        en=payload.get("source_text", ""),
                        tgt=payload.get("target_text", ""),
                        source_lang=payload.get("source_lang", source_lang),
                        target_lang=payload.get("target_lang", target_lang),
                        score=hit.score,
                    )
                )
            logger.debug("Vector search returned %d results", len(results))
            return results
        except Exception as e:
            logger.error("Vector search failed: %s", e)
            return []


_vector_index = None


def get_vector_index() -> VectorIndex:
    global _vector_index
    if _vector_index is None:
        _vector_index = VectorIndex()
    return _vector_index
```

### Разбор ключевых решений

**Почему `_POINT_NAMESPACE = uuid.UUID("6ba7b810-9dad-11d1-80b4-00c04fd430c8")`.** Это `uuid.NAMESPACE_DNS` — стандартный namespace для DNS-имён (RFC 4122). Выбран произвольно; важен только факт, что **один** namespace на весь проект. Если бы использовали `uuid.uuid4()` каждый раз — разные namespace для разных процессов, point id не совпадали бы между перезапусками.

**Почему `uuid5` (SHA-1-based), а не `uuid4` (random).** `uuid5(namespace, name)` — детерминированный: один `key` → один UUID. Значит, `upsert` одной и той же записи перезаписывает точку, а не создаёт дубликат. `uuid4` — случайный, каждый `add_entries` создаёт новый UUID, дубликаты неизбежны. При повторном импорте CSV (Этап 6) `add_many` вызывает `add_entries` с теми же данными — с `uuid5` это no-op (upsert того же point id), с `uuid4` — N новых точек.

**Почему `uuid5` (SHA-1), а не `uuid5` с другим хешем.** Python stdlib: `uuid5` — SHA-1, `uuid3` — MD5 (deprecated в криптоконтексте, но детерминизм тот же). `uuid5` — правильный выбор для детерминированных идентификаторов; SHA-1 коллизии требуют 2⁸⁰ вычислений, для наших данных (тысячи записей) — не проблема.

**Почему `key = f"{source_lang}|{target_lang}|{en}|{tgt}"` — полный контент.** Два разных перевода одного source (`en="hello"` → `tgt="привет"` и `tgt="здравствуй"`) должны быть **разными** точками. Включение `tgt` в key разделяет их. Если бы `key = f"{source_lang}|{target_lang}|{en}"` — оба перевода получили бы один point id, второй upsert перезаписал бы первый. Разные переводы — разные точки.

**Почему `key` для `uuid5` **идентичен** входу для `_entry_id` в `TMManager`, но результат разный.** Оба детерминированы, оба derived from `source_lang|target_lang|en|tgt`. `_entry_id` = `<src>|<tgt>|<sha256>` (B-tree-safe строка для Postgres). `_point_id` = `<uuid5>` (валидный UUID для Qdrant). Оба нужны, оба консистентны.

**Почему `client_kwargs["check_compatibility"] = False`.** Qdrant-клиент по умолчанию проверяет версию сервера при подключении. Если версия клиента и сервера расходятся на мажорную — `RuntimeError: Compatibility check failed`. `check_compatibility=False` отключает проверку: клиент работает, даже если сервер новее. Для учебного проекта — правильный выбор (не хотим ловить несовместимость между `qdrant/qdrant:latest` и `qdrant-client ^1.19.0`).

**Почему `client_kwargs["timeout"] = 10.0`.** Запросы к Qdrant — быстрые (< 100 мс на локальном сервере). 10 секунд — запас на холодный старт и сетевые задержки. Не 300, как у LLM: векторный поиск не должен занимать минуты.

**Почему `_ensure_connection_and_collection(retries=5, delay=2)`.** Qdrant в Docker стартует 5–10 с. `app` тоже стартует, но с `depends_on: qdrant: service_healthy`. Тем не менее, `depends_on` гарантирует только «healthcheck прошёл», а не «REST API готов». 5 попыток × 2 с = 10 с — покрывает окно инициализации. `time.sleep(delay)` — синхронный, потому что `_ensure_connection_and_collection` вызывается из sync-методов (`add_entries`, `search`). В async-контексте `time.sleep` блокировал бы event loop — но `_ensure_*` вызывается из `to_thread` (в `TMManager`), так что безопасно.

**Почему `self.client = QdrantClient(**client_kwargs)` при каждой попытке.** После неудачи `self.client = None`, при следующей итерации — новый клиент. Если бы использовали один клиент без сброса — он мог бы остаться в «полу-подключенном» состоянии. Новый клиент на каждую попытку — чисто.

**Почему `self.client.get_collections()` внутри try, а не `self.client.info()`.** `get_collections()` — лёгкий запрос, проверяет auth + доступ к метаданным. `info()` возвращает больше данных, но тоже работает. `get_collections` — стандартный «ping».

**Почему `_ensure_collection` вызывается **после** `get_collections` в `_ensure_connection_and_collection`.** Сначала проверяем, что клиент вообще подключился. Если `get_collections` упал — `_ensure_collection` не вызываем (клиент `None`, будет `AttributeError`).

**Почему `_ensure_collection` не пересоздаёт существующую.** `self.client.get_collection(name)` бросает `ValueError`/`NotFound`, если коллекции нет. `except Exception` — создаём. Если коллекция есть — `logger.debug` и выход. **Не** пересоздаём: два FastAPI-воркера, оба вызывают `_ensure_*` при первом запросе, оба пытались бы пересоздать — race condition, данные теряются.

**Почему `VectorParams(size=embedding_model.dimension, distance=Distance.COSINE)`.** `dimension` — 384 для `paraphrase-multilingual-MiniLM-L12-v2`. Если модель поменяется, dimension поменяется, коллекция станет несовместимой (Qdrant проверяет размерность при upsert). Для учебного проекта — фиксированная модель, размерность константна.

**Почему `recreate_collection` — отдельный метод, а не флаг в `_ensure_collection`.** `recreate_collection` — опасная операция (drop данных), вызывается **только** из Celery-задачи `index_translation_memory` (single-process). Если бы `_ensure_collection` имел флаг `recreate=True`, любой вызов мог бы случайно передать его и стереть данные. Отдельный метод — explicit control.

**Почему `add_entries` собирает `texts = [entry.en for entry in entries]` и делает **один** `embedding_model.encode(texts)`, а не `encode` в цикле.** `SentenceTransformer.encode` батчит внутри (по 32), а на GPU делает один forward pass для всего батча. `encode` в цикле — N forward passes, медленно. Один вызов на N записей — один batched forward pass. Разница для 1000 записей на CPU: ~30 с vs ~5 минут.

**Почему `payload["id"] = entry.id` только если `entry.id is not None`.** При импорте из CSV (Этап 6) `TMEntry.id` может быть `None` (caller не передал). При загрузке из БД (`_fetch_all_entries`) — `id` всегда есть. Если `None`, пропускаем — Qdrant payload без `id` валиден, но `search` вернёт `id=None`. Caller должен учитывать.

**Почему `search` использует `query_points`, а не `search`.** `qdrant-client 1.19` использует `query_points` как основной API. Старый `search` удалён. `query_points` возвращает `QueryResponse` с `.points: List[ScoredPoint]`, каждый — `.id`, `.score`, `.payload`.

**Почему `search` фильтрует только по `source_lang` и `target_lang`, а не по `source_text`.** При поиске мы ищем ближайший перевод для **нового** source_text, которого в базе нет. Фильтр по source_text был бы пустым (не найдёт себя же). Фильтр по языкам — правильно: ищем в той же языковой паре.

**Почему `delete_by_filter` фильтрует по **всем** 4 полям.** Source_text и target_text могут быть одинаковыми у разных пар (например, «hello» → «привет» для en→ru и en→ru второй раз). Но `id` в payload уникален по контенту, а filter по тексту — находит все точки с этими текстами. Для `delete_entry(entry_id)` мы знаем **точный** entry; filter по 4 полям гарантированно находит одну точку (или ноль).

**Почему `delete_by_filter` с retry (3 попытки × 1 с).** Qdrant может быть временно перегружен, или сетевая задержка. Одна попытка недостаточна. Три — компромисс. Если все три упали — лог `error`, но не бросаем: `delete_entry` в `TMManager` уже удалил из SQL и `_tm_cache`, Qdrant остался с «мертвой» точкой — следующий `recreate_collection` (Celery) её уберёт.

**Почему `search` возвращает `score=hit.score`, а не `score=None`.** Qdrant возвращает `score` для каждой точки (cosine similarity). Это полезно для отладки: видно, насколько близко. `TMEntry.score: Optional[float]` — принимает.

**Почему `get_vector_index()` — фабрика, а не module-level singleton.** `VectorIndex()` при импорте попытался бы подключиться к Qdrant (через `_ensure_*`). Модуль импортируется в тестах без Qdrant — падал бы. Фабрика — первый вызов `get_vector_index()` создаёт `VectorIndex()`, но подключение происходит при первом `add_entries`/`search`, не при создании.

**Почему `_vector_index = None` глобальная, а не class-variable.** `VectorIndex()` — один экземпляр на процесс. Class-variable работала бы, но глобальная переменная проще для monkeypatch в тестах: `monkeypatch.setattr("backend.retrieval.vector_index._vector_index", fake)`.

### Коммит

```powershell
git add backend/retrieval/vector_index.py
git commit -m "feat(week-7): add Qdrant VectorIndex with uuid5 point IDs"
```

---

## 7.5. `backend/retrieval/hybrid_retriever.py` — RRF

**Что это:** `reciprocal_rank_fusion(bm25_results, vector_results, k=60)` — объединяет два ранжированных списка в один. `hybrid_search(query, src, tgt, top_n)` — параллельно вызывает BM25 и vector, объединяет.

**Зачем это нужно:** BM25 и vector возвращают **разные** результаты. BM25 — точные лексические совпадения, vector — семантические. RRF объединяет их, давая каждому «голос» по позиции в рейтинге.

Создайте `backend/retrieval/hybrid_retriever.py`:

```python
# backend/retrieval/hybrid_retriever.py
import asyncio
import logging
from typing import Dict, List, Optional, Tuple

from backend.core.models import TMEntry
from backend.data.translation_memory import tm_manager
from backend.retrieval.vector_index import get_vector_index

logger = logging.getLogger(__name__)

_EntryKey = Tuple[str, str, str, str]


def reciprocal_rank_fusion(
    bm25_results: List[TMEntry],
    vector_results: List[TMEntry],
    k: int = 60,
) -> List[TMEntry]:
    """
    Reciprocal Rank Fusion over two ranked lists.

    Scores accumulate as 1 / (k + rank + 1) per source. The `id` of a
    fused entry is taken from whichever source provided it — BM25
    (populated by ``TMManager._fetch_all_entries``) or vector search
    (populated from the Qdrant payload). If neither carries an id
    (legacy points written before the id backfill), the fused entry
    keeps ``id=None``.
    """
    scores: Dict[_EntryKey, float] = {}
    ids: Dict[_EntryKey, Optional[str]] = {}

    for rank, entry in enumerate(bm25_results):
        key = (entry.en, entry.tgt, entry.source_lang, entry.target_lang)
        scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank + 1)
        if entry.id is not None and key not in ids:
            ids[key] = entry.id

    for rank, entry in enumerate(vector_results):
        key = (entry.en, entry.tgt, entry.source_lang, entry.target_lang)
        scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank + 1)
        if entry.id is not None and key not in ids:
            ids[key] = entry.id

    sorted_keys = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    results: List[TMEntry] = []
    for key, score in sorted_keys:
        en, tgt, src_lang, tgt_lang = key
        results.append(TMEntry(
            id=ids.get(key),
            en=en,
            tgt=tgt,
            source_lang=src_lang,
            target_lang=tgt_lang,
            score=score,
        ))
    return results


async def hybrid_search(
    query: str,
    source_lang: str,
    target_lang: str,
    top_n: int = 5,
) -> List[TMEntry]:
    bm25_results = await tm_manager.search(
        query, source_lang, target_lang, top_n=top_n * 2
    )

    try:
        vector_results = await asyncio.to_thread(
            get_vector_index().search,
            query,
            source_lang,
            target_lang,
            top_n=top_n * 2,
        )
    except Exception as e:
        logger.warning("Vector search failed, using BM25 only: %s", e)
        vector_results = []

    fused = reciprocal_rank_fusion(bm25_results, vector_results)
    return fused[:top_n]
```

### Разбор ключевых решений

**Почему `k=60`.** Классическое значение RRF из оригинальной работы Cormack et al. (2009). `k` сглаживает вклады: `1/(60+rank+1)` даёт `1/61 ≈ 0.0164` для rank 0, `1/62 ≈ 0.0161` для rank 1. Разница между топ-1 и топ-2 мала. Если `k=0`, `1/(rank+1)` даёт резкое падение (1.0 vs 0.5) — топ-1 доминирует. `k=60` — баланс: топ-1 не давит остальные, но и не сливается с серединой.

**Почему `1/(k + rank + 1)`, а не `1/(k + rank)`.** `rank` с 0, `k=60` → `60+0+1 = 61` (не 60). Сдвиг на 1 гарантирует, что `k=0` даёт `1/(rank+1)` (без деления на ноль для rank=0). Косметика, но стандарт.

**Почему `key = (en, tgt, source_lang, target_lang)` — 4-tuple.** Дедупликация: одна и та же TM-запись из BM25 и vector имеет одинаковый `en`/`tgt`/`source_lang`/`target_lang`. Если её нашли оба — суммируем score (это и есть цель RRF: «два источника согласны, это важнее»). Если `key = entry.id` — работало бы только при наличии `id` в обоих, а vector иногда возвращает `id=None` (старые точки). 4-tuple работает всегда.

**Почему `scores.get(key, 0.0) + 1.0 / (k + rank + 1)`.** Если ключа нет — начинаем с 0. Если есть (пришёл из другого списка) — прибавляем. `scores.get(key, 0.0)` — безопасно.

**Почему `ids[key] = entry.id` только если `key not in ids`.** Первый источник, где есть `id`, выигрывает. BM25 приходит первым (`for bm25_results` до `for vector_results`) — BM25 `id` предпочтительнее. Оба должны быть одинаковыми (`_entry_id` детерминирован), но если vector вернул старую точку с другим `id` — BM25 правильный.

**Почему `sorted(scores.items(), key=lambda x: x[1], reverse=True)`.** Сортировка по score. Для tie — Python stable sort сохраняет порядок вставки (BM25 первым). Не идеально для tie, но tie редки (float-сложение), и любое разрешение приемлемо.

**Почему `hybrid_search` запрашивает `top_n * 2` из каждого источника.** RRF объединяет два списка по `top_n * 2` записей, потом обрезает до `top_n`. Если бы запрашивали `top_n` — после дедупликации могло бы остаться меньше `top_n` (если пересечение большое). `* 2` даёт запас: в худшем случае (полное пересечение) останется `top_n` уникальных, в лучшем — `top_n` из объединённого топа.

**Почему `asyncio.to_thread(get_vector_index().search, ...)`.** `VectorIndex.search` — синхронный (qdrant-client sync). `to_thread` выносит в threadpool, event loop свободен. `tm_manager.search` — async, вызывается напрямую.

**Почему `try/except` вокруг `get_vector_index().search`, но не вокруг `tm_manager.search`.** `tm_manager.search` — BM25, in-memory, не бросает (`try/except` внутри). `vector_index.search` — сетевой вызов к Qdrant, может упасть (`ConnectionError`, `TimeoutError`). При падении — `vector_results = []`, RRF с одним списком = BM25. Это и есть «деградация до BM25».

**Почему `fused[:top_n]`, а не `fused[:top_n]` с фильтром.** RRF может вернуть до `len(bm25) + len(vector)` уникальных, обычно `~2 * top_n`. Обрезка до `top_n` — правильное поведение для caller'а.

**Почему `_EntryKey = Tuple[str, str, str, str]` — type alias.** Читаемость: `Dict[_EntryKey, float]` понятнее, чем `Dict[Tuple[str, str, str, str], float]`. Плюс если структура ключа поменяется — один alias, а не три сигнатуры.

**Почему `id=ids.get(key)`, а не `id=ids[key]`.** В `ids` могут быть не все ключи из `scores` (если entry был из vector с `id=None` и не из BM25). `ids.get(key)` → `None`. `ids[key]` бросил бы `KeyError`.

### Проверка

```powershell
python -m poetry run python -c "
from backend.retrieval.hybrid_retriever import reciprocal_rank_fusion
from backend.core.models import TMEntry
a = [TMEntry(id='1', en='hello', tgt='привет', source_lang='en', target_lang='ru', score=2.5)]
b = [TMEntry(id='1', en='hello', tgt='привет', source_lang='en', target_lang='ru', score=0.9)]
fused = reciprocal_rank_fusion(a, b)
print('fused count:', len(fused))
print('score:', fused[0].score)
"
```

**Что вы должны увидеть** (score = `1/(60+0+1) + 1/(60+0+1) ≈ 0.0328`):

```
fused count: 1
score: 0.03278688524590164
```

### Коммит

```powershell
git add backend/retrieval/hybrid_retriever.py
git commit -m "feat(week-7): add RRF and hybrid_search"
```

---

## 7.6. `infrastructure/docker-compose.yml` — сервис `qdrant`

**Что это:** добавить в `docker-compose.yml` (создан в Этапе 3) сервис `qdrant` с named volume, healthcheck и зависимостью `app`.

**Зачем это нужно:** без Qdrant `VectorIndex.add_entries` 5 раз пробует подключиться (10 секунд), логирует warning, пропускает. Данные в Qdrant отсутствуют — hybrid-режимы в Этапе 9 работают в режиме «BM25-only», теряя семантический поиск.

Откройте `infrastructure/docker-compose.yml` и добавьте:

**1. Сервис `qdrant` — после `redis` (или после `db`, порядок не важен):**

```yaml
  qdrant:
    image: qdrant/qdrant:latest
    volumes:
      - qdrant_storage:/qdrant/storage
    restart: unless-stopped
    healthcheck:
      # Debian 13 image, no wget/curl available. Opening a TCP socket
      # to the REST port is enough to confirm the process is up and
      # listening. 127.0.0.1 forces IPv4 — ``localhost`` may resolve to
      # ::1 inside the container, which Qdrant does not bind by default.
      test: ["CMD", "bash", "-c", ":> /dev/tcp/127.0.0.1/6333"]
      interval: 10s
      timeout: 5s
      retries: 5
      start_period: 10s
    networks:
      - rung_internal
```

**2. В `app.depends_on` — добавить `qdrant`:**

```yaml
    depends_on:
      db:
        condition: service_healthy
      qdrant:
        condition: service_healthy
```

**3. В `volumes:` — добавить `qdrant_storage`:**

```yaml
volumes:
  postgres_data:
    name: rung_postgres_data
  qdrant_storage:
    name: rung_qdrant_storage
```

### Разбор ключевых решений

**Почему `image: qdrant/qdrant:latest`, а не фиксированная версия.** В референсном коде — `latest`. Для production лучше `qdrant/qdrant:v1.12.0` (фиксированная), но для учебного проекта `latest` работает: `qdrant-client ^1.19.0` совместим с большинством версий Qdrant (проверка `check_compatibility=False` отключает строгий матчинг).

**Почему healthcheck через `bash -c ":> /dev/tcp/127.0.0.1/6333"`.** В образе `qdrant/qdrant:latest` (Debian 13) нет `wget` и `curl`. Остаются:
- `pg_isready`-аналога нет.
- `bash` есть — встроенная поддержка `/dev/tcp/`.

`:> /dev/tcp/host/port` — открывает TCP-сокет, пишет пустую строку, закрывает. Если порт слушается — exit 0. Если нет — exit 1. Это liveness-проверка: процесс жив и слушает.

**Почему `127.0.0.1`, а не `localhost`.** `localhost` может резолвиться в `::1` (IPv6). Qdrant по умолчанию слушает только IPv4 (`0.0.0.0:6333`), не `[::]:6333`. `bash` с `/dev/tcp/localhost/6333` мог бы пойти в `::1` и упасть с `Connection refused`, хотя сервис жив. `127.0.0.1` — force IPv4.

**Почему порт `6333` (REST), а не `6334` (gRPC).** `QdrantClient(url=...)` использует REST. gRPC — быстрее, но требует `prefer_grpc=True` и другого конфига. Для учебного проекта REST проще. Healthcheck на REST-порт соответствует тому, что использует клиент.

**Почему `start_period: 10s`.** Qdrant стартует ~5 с (загрузка RocksDB, восстановление коллекций). 10 с — grace period: если сервис не отвечает в первые 10 с, это не считается провалом healthcheck. После `start_period` — проверки каждые `interval: 10s`, при `retries: 5` провалах — unhealthy.

**Почему `restart: unless-stopped`.** Qdrant может упасть от OOM при большой коллекции. `unless-stopped` — автоматический рестарт, но не после `docker compose stop qdrant` (явной остановки).

**Почему named volume `rung_qdrant_storage`, а не bind mount.** Docker-managed volume переживает `docker compose down`, но не `docker compose down -v`. Данные (индексы, точки) сохраняются между перезапусками. Bind mount (`./data/qdrant:/qdrant/storage`) тоже работал бы, но требует прав доступа (Qdrant запускается от uid 1000).

**Почему `depends_on: qdrant: service_healthy` у `app`, а не просто `condition: service_started`.** Qdrant стартует ~5 с, `app` — ~10–15 с (Python + uvicorn). `service_started` даёт `app` стартовать сразу, как только контейнер `qdrant` создан, но до того, как Qdrant-процесс начал слушать порт. `app` при первом `_ensure_connection_and_collection` получил бы `Connection refused` 5 раз (10 с), потом `client = None`, warning. С `service_healthy` `app` ждёт, пока Qdrant реально готов.

**Почему `service_healthy` важнее для Qdrant, чем для `db`.** `db` (Postgres) стартует ~5 с, но `pg_isready` проверяет именно готовность принимать соединения. Qdrant без healthcheck считался бы «запущенным» сразу после `docker start`, но REST API ещё не готов. Healthcheck — обязателен.

### Запуск

```powershell
cd D:\RUNG\infrastructure
docker compose up -d --build
```

**Что вы должны увидеть:**

```
[+] Running 5/5
 ✔ Network rung_internal          Created
 ✔ Volume "rung_postgres_data"    Created
 ✔ Volume "rung_qdrant_storage"   Created
 ✔ Container infrastructure-db-1  Healthy
 ✔ Container infrastructure-qdrant-1  Healthy
 ✔ Container infrastructure-app-1 Started
```

`qdrant` перешёл в `Healthy` (healthcheck прошёл). `app` — `Started`.

Проверьте:

```powershell
docker compose ps
```

**Что вы должны увидеть:**

```
NAME                       SERVICE   STATUS                    PORTS
infrastructure-db-1        db        Up 30 seconds (healthy)   5432/tcp
infrastructure-qdrant-1    qdrant    Up 15 seconds (healthy)   6333/tcp, 6334/tcp
infrastructure-app-1       app       Up 10 seconds (healthy)   0.0.0.0:8000->8000/tcp
```

`qdrant` — в списке. Порты `6333`/`6334` внутри сети `rung_internal` (наружу не проброшены — только `app` их использует).

### Проверка Qdrant изнутри

```powershell
docker compose exec qdrant wget -qO- http://127.0.0.1:6333/collections
```

**Что вы должны увидеть:**

```json
{"result":{"collections":[]},"status":"ok","time":0.0001}
```

Пустой список — коллекций ещё нет. После первого `add_entries` появится `tm_vectors`.

### Эксперимент: переиндексировать TM в Qdrant

Данные из Этапа 6 уже в БД, но в Qdrant их нет (Этап 6 писал в warning «Qdrant unavailable»). Переиндексируйте:

```powershell
python -m poetry run python -c "
import asyncio
from backend.data.translation_memory import tm_manager
async def m():
    await tm_manager.load_all_into_cache()
    print('done')
asyncio.run(m())
"
```

**Что вы должны увидеть** (второй запуск, Qdrant уже поднят):

```
... INFO Connected to Qdrant at http://qdrant:6333
... INFO Qdrant collection 'tm_vectors' ready
... INFO Rebuilt all indexes for N entries
done
```

Проверьте коллекцию:

```powershell
docker compose exec qdrant wget -qO- http://127.0.0.1:6333/collections/tm_vectors
```

**Что вы должны увидеть:**

```json
{"result":{"status":"green","optimizer_status":"ok","vectors_count":N,"points_count":N,...},"status":"ok"}
```

`points_count: N` — N точек в Qdrant, совпадает с `COUNT(*)` в `translation_memories`.

### Коммит

```powershell
git add infrastructure/docker-compose.yml
git commit -m "feat(week-7): add qdrant service with TCP healthcheck"
```

---

## 7.7. Тесты

Тесты скинет пользователь отдельно. Здесь — что должно проверяться.

### `tests/unit/test_bm25.py`

- `initialize([])` → `search` возвращает `[]`.
- `initialize` 5 записей, `search("hello", top_n=3)` → 1–3 записи с `score > 0`.
- `search` для запроса, которого нет → `[]` (все scores = 0).
- `search` возвращает `TMEntry` с заполненным `score`.
- `search("")` → `[]`.
- `search("  ")` → `[]` (после strip).

### `tests/unit/test_tm_search.py`

- `TMManager._bm25_dirty` пуст после `ensure_loaded`.
- `add_entry` добавляет пару в `_bm25_dirty`, но **не** пересобирает BM25 сразу.
- `search` для грязной пары → пересобирает, `_bm25_dirty` очищается.
- `search` для чистой пары → не пересобирает.
- `add_many` с 100 записями → 1 пересборка при следующем `search`, не 100.
- `delete_entry` → добавляет пару в `_bm25_dirty`, `search` пересобирает.
- `insert_in_session` с дубликатом → `inserted` содержит только уникальные.

### `tests/unit/test_rrf.py`

- `reciprocal_rank_fusion([], [])` → `[]`.
- `reciprocal_rank_fusion([e1], [])` → `[e1]` с score `1/61`.
- `reciprocal_rank_fusion([], [e2])` → `[e2]` с score `1/61`.
- `reciprocal_rank_fusion([e1], [e1])` (одинаковые) → `[e1]` с score `2/61`.
- `reciprocal_rank_fusion([e1, e2], [e2, e3])` → `[e2, e1, e3]` (e2 на первом месте — в обоих списках).
- `id` берётся из BM25 при конфликте.
- `id=None`, если ни в одном источнике нет `id`.
- `k=60` по умолчанию.

### `tests/unit/test_hybrid_search.py`

Mock Qdrant + реальный BM25:
- `hybrid_search` с mock `vector_index.search` → `[]`, BM25 вернул результаты → RRF с одним списком = BM25.
- `hybrid_search` с mock `vector_index.search`, бросающим `ConnectionError` → warning, BM25-only, без падения.
- `hybrid_search` с обоими → RRF объединяет.

**Mock `vector_index.search`.** `monkeypatch.setattr("backend.retrieval.hybrid_retriever.get_vector_index", lambda: MockVectorIndex())`, где `MockVectorIndex.search` возвращает заданный список.

**Mock `tm_manager.search`.** `monkeypatch.setattr("backend.retrieval.hybrid_retriever.tm_manager.search", AsyncMock(return_value=[...]))`.

**Фикстура `autouse` для `unload_hf_llm`.** Если тесты загружают embeddings (BM25 тесты не должны), убедитесь, что `embedding_model` не загружается в тестах без необходимости. `test_bm25` использует `BM25Index` — `embedding_model` не нужен. `test_hybrid_search` мокает `vector_index` — `embedding_model` тоже не нужен. Единственный тест, который загрузит модель — это `test_embedding_model`, если вы его добавите.